In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(loomR)
library(SCopeLoomR)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep25/"
dataDir <- paste0(mainDir, "data/")
repDir <- paste0(mainDir, "pyscenic/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
### Load object
seu <- readRDS(paste0(mainDir, "QC_clustering/final_clusters.rds"))

In [ ]:
### Get top variable genes
DefaultAssay(other_subset) <- "RNA"

if (length(VariableFeatures(other_subset)) == 0) {
  other_subset <- NormalizeData(other_subset)
  other_subset <- FindVariableFeatures(other_subset, selection.method = "vst", nfeatures = 5000)
}

top5000_genes <- intersect(VariableFeatures(other_subset), rownames(other_subset))

In [ ]:
### Subset seurat object to top variable genes by rebuilding object
valid_genes <- Reduce(intersect, list(
  rownames(LayerData(other_subset, layer = "counts")),
  rownames(LayerData(other_subset, layer = "data")),
  top5000_genes
))

other_subset_5000 <- CreateSeuratObject(
  counts = LayerData(other_subset, layer = "counts")[valid_genes, ],
  data = LayerData(other_subset, layer = "data")[valid_genes, ]
)

other_subset_5000@meta.data <- other_subset@meta.data
VariableFeatures(other_subset_5000) <- valid_genes

In [ ]:
columns_to_keep <- c("sample", "condition", "genotype", "timepoint", "seurat_clusters") 
other_subset_5000@meta.data <- other_subset_5000@meta.data[, columns_to_keep]

# Copy UMAP from original object
if ("umap" %in% Reductions(other_subset)) {
  other_subset_5000[["umap"]] <- other_subset[["umap"]]
}

In [ ]:
### Export to Scope compatible loom file ready for pyscenic
loom_file <- paste0(repDir, "2025.loom")
counts_matrix <- GetAssayData(seu, assay = "RNA", slot = "counts")
umap_embeddings <- Embeddings(seu, reduction = "umap")
build_loom(
  file.name = loom_file,
  dgem = as.matrix(counts_matrix),
  title = "2025 N+BreRi, N+prosRi, N+wRi, prosRi",
  default.embedding = umap_embeddings,
  default.embedding.name = "UMAP"
)
loom <- open_loom(loom_file, mode = "r")
print(loom)
close_loom(loom)

In [ ]:
auc_matrix <- read.csv(paste0(repDir, "pyscenic_res_5000_3/run__mean.csv"), row.names = 1, check.names = FALSE)
regulons <- read.csv("/data/ebaird/scRNAseq/scripts/consistent_tfs_regulons.csv", check.names = FALSE)

In [ ]:
regulons_names <- colnames(regulons)

In [ ]:
### Post-scenic analysis
# scenic_loom <- connect(
#   filename = "/data/ebaird/scRNAseq/SCENTINELsep24/downstream/Gal10d_Gal12d_Flp10d_Flp12d/pyscenic_res_3000slr/output.loom", 
#   mode = "r",
#   skip.validate = TRUE
# )

# auc_matrix <- scenic_loom$col.attrs$RegulonsAUC[]
# auc_matrix <- read.csv(paste0(repDir, "pyscenic_res_5000_3/run__mean.csv"))

# rownames(auc_matrix) <- scenic_loom$col.attrs$CellID[]

# scenic_loom$close()

# auc_matrix <- as.matrix(auc_matrix)

common_cells <- intersect(rownames(auc_matrix), colnames(seu))
auc_matrix <- auc_matrix[common_cells, ]

# regulons <- read.csv(paste0(repDir, "pyscenic_res_5000_3/consistent_tfs_regulons.csv"))

# ## OPTION A: As metadata (simpler)
# seu <- AddMetaData(
#   object = seu,
#   metadata = as.data.frame(auc_matrix)
# )

## OPTION B: As new assay (better for visualization)
seu[["regulons"]] <- CreateAssayObject(
  counts = t(auc_matrix)
)

DefaultAssay(seu) <- "regulons"

all_regulons <- rownames(seu[["regulons"]])

seu <- ScaleData(seu)
seu <- RunPCA(seu, features = all_regulons, reduction.name = "pca_regulons")

#elbow plot
ElbowPlot(seu, reduction = "pca_regulons", ndims = 30) +
  ggtitle("Elbow Plot for Regulons PCA")

In [ ]:
seu@meta.data$original_clusters <- seu@meta.data$seurat_clusters

seu <- FindNeighbors(seu, reduction = "pca_regulons", dims = 1:10)
seu <- FindClusters(seu, graph.name = "regulons_snn", resolution = 0.5)

seu <- RunUMAP(seu, reduction = "pca_regulons", dims = 1:10, reduction.name = "umap_regulons")
DimPlot(seu, reduction = "umap_regulons", group.by = "regulons_snn_res.0.5")

In [ ]:
DefaultAssay(seu) <- "RNA"
DimPlot(seu, reduction = "umap_regulons", group.by = "regulons_snn_res.0.5")
DimPlot(seu, reduction = "umap_regulons", group.by = "original_clusters", label=FALSE)
DimPlot(seu, reduction = "umap", group.by = "regulons_snn_res.0.5", label=FALSE)

### Save plots
jpeg(paste0(figDir, 'umap_regulons_10.07.jpeg'), quality = 100, width = 1000, height = 1000, res = 150)
DimPlot(seu, reduction = "umap_regulons", group.by = "regulons_snn_res.0.5") +
  ggtitle("Regulons UMAP by Regulon Clusters")
dev.off()
jpeg(paste0(figDir, 'umap_regulons_original_clusters_10.07.jpeg'), quality = 100, width = 1000, height = 1000, res = 150)
DimPlot(seu, reduction = "umap_regulons", group.by = "original_clusters", label=FALSE) +
  ggtitle("Regulons UMAP by Original Clusters")
dev.off()
jpeg(paste0(figDir, 'umap_seurat_regulon_clusters_10.07.jpeg'), quality = 100, width = 1000, height = 1000, res = 150)
DimPlot(seu, reduction = "umap", group.by = "regulons_snn_res.0.5", label=FALSE) +
  ggtitle("Original UMAP by Regulon Clusters")
dev.off()

In [ ]:
### differential expression analysis for regulons
# Idents(seu) <- seu$regulons_snn_res.0.5
# degs_regulons <- FindAllMarkers(seu, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
# write.csv(degs_regulons, file = paste0(tabDir, "degs_regulons_10.07.csv"))
# top10 <- degs_regulons %>%
#   group_by(cluster) %>%
#   top_n(n = 10, wt = avg_log2FC)
# write.csv(top10, file = paste0(tabDir, "top10_degs_regulons_10.07.csv"))

# Plot dotplot of top 10 DEGs per regulon cluster
top10_genes <- unique(top10$gene)
jpeg(paste0(figDir, 'top10markers.dotplot_regulon_clusters.jpeg'), quality = 100, width = 2000, height = 1000, res = 150)
print(
  DotPlot(seu, features = top10_genes, dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
    limits = c(0, 1.5),
    oob = scales::squish
  ) +
  ggtitle("Top 10 Markers per regulon cluster")
)
dev.off()

In [ ]:
# saveRDS(seu, file = paste0(repDir, "seurat_with_regulons.rds"))
seu <- readRDS(paste0(repDir, "seurat_with_regulons.rds"))

In [ ]:
# Feature plot for regulons
DefaultAssay(seu) <- "regulons"

plots <- list()

for (regulon in regulons_names) {
  p <- FeaturePlot(seu, features = regulon, reduction = "umap_regulons", label = FALSE) +
    ggtitle(paste("Feature Plot for Regulon:", regulon)) +
    scale_colour_gradientn(colours = cols)
  plots[[regulon]] <- p
}

ggexport(ggarrange(plotlist = plots, ncol = 3, nrow = ceiling(length(plots) / 3)), 
         filename = paste0(figDir, "feature_plots_all_regulons_regulon_umap_10.07.jpeg"), 
         width = 2000, height = 4000, res = 150)


plots <- list()

for (regulon in regulons_names) {
  p <- FeaturePlot(seu, features = regulon, reduction = "umap", label = FALSE) +
    ggtitle(paste("Feature Plot for Regulon:", regulon)) +
    scale_colour_gradientn(colours = cols)
  plots[[regulon]] <- p
}

ggexport(ggarrange(plotlist = plots, ncol = 3, nrow = ceiling(length(plots) / 3)), 
         filename = paste0(figDir, "feature_plots_all_regulons_seurat_umap_10.07.jpeg"), 
         width = 2000, height = 4000, res = 150)


In [ ]:
# other visualisations dotplot, heatmap

# p <- DoHeatmap(seu, features = all_regulons, group.by = "original_clusters") +
#   ggtitle("Heatmap for Regulons")

# ggsave(filename = paste0(figDir, "heatmap_regulons.pdf"), plot = p, width = 8, height = 6)

# p <- DotPlot(seu, features = all_regulons, group.by = "original_clusters") +
#   ggtitle("Dotplot for Regulons") +
#   theme(
#     axis.text.x = element_text(size = 8)
#   ) +
#   RotatedAxis() +
#   scale_color_gradient(
#     low    = "white",
#     high   = "forestgreen",
#     limits = c(0, 1.5),
#     oob    = scales::squish)
  

# ggsave(filename = paste0(figDir, "dotplot_regulons.pdf"), plot = p, width = 8, height = 6)

# Dotplot for regulon TF
regulon_TF <- c(
  "BtbVII","vvl","Pdp1"
)

DefaultAssay(seu) <- "SCT"
p <- DotPlot(seu, features = regulon_TF, group.by = "original_clusters") +
  ggtitle("Dotplot for vvl regulon TFs") +
  theme(
    axis.text.x = element_text(size = 8)
  ) +
  RotatedAxis() +
  scale_color_gradient(
    low    = "white",
    high   = "forestgreen",
    limits = c(0, 1.5),
    oob    = scales::squish)
ggsave(filename = paste0(figDir, "dotplot_vvl_regulon_TF.pdf"), plot = p, width = 8, height = 6)


In [ ]:
### Export to Scope compatible loom file with scenic results
loom_file <- paste0(repDir, "seurat_with_regulons.loom")
counts_matrix <- GetAssayData(seu, assay = "RNA", slot = "counts")
umap_embeddings <- Embeddings(seu, reduction = "umap")
build_loom(
  file.name = loom_file,
  dgem = as.matrix(counts_matrix),
  title = "Seurat with regulons",
  default.embedding = umap_embeddings,
  default.embedding.name = "UMAP"
)
loom <- open_loom(loom_file, mode = "r")
print(loom)
close_loom(loom)

In [ ]:
loom <- open_loom(loom_file, mode = "r+")

auc_matrix <- as.matrix(GetAssayData(seu, assay = "regulons", slot = "data"))
regulons <- regulon_list
common_regulons <- intersect(rownames(auc_matrix), names(regulons))
if(length(common_regulons) == 0) stop("No matching regulons found between AUC matrix and gene sets")
auc_matrix <- auc_matrix[common_regulons, ]
regulons <- regulons[common_regulons]

# Add regulons to loom file
add_regulons(
  loom = loom,
  regulons = regulons,
  regulonAUC = auc_matrix
)

# Close loom connection
close_loom(loom)

# Verify file contents
final_loom <- open_loom(loom_file)
print(final_loom)
close_loom(final_loom)